# 5. `Fan speeds ( <N> <N> <N> <N> <N> <N> )` (switch_module, fan)

In [1]:
from common import *

shape, log = show(5)

log shape 38: switch_module | fan | Fan speeds ( <N> <N> <N> <N> <N> <N> )
21,559 logs (5.2%) from 1 nodes, 2004-09-19 04:42:40 to 2006-04-10 06:03:16
log 50422 at 2005-09-05 14:56:30: 41953 Interconnect-0N00 switch_module fan 1125932190 1 Fan speeds ( 3552 3534 3375 **** 3515 3479 )


In [2]:
print("I'd say that if the speeds were close to zero this should be a concern.")
print("Five of the six are above 1000 RPM (guessing a normal minimum speed here), but the fourth is '****': that fan has no reading.")
print("Should we worry about the fan with no reading?")

I'd say that if the speeds were close to zero this should be a concern.
Five of the six are above 1000 RPM (guessing a normal minimum speed here), but the fourth is '****': that fan has no reading.
Should we worry about the fan with no reading?


Every fan report of the 7 days before, one column per fan

In [8]:
reports = as_frame(logs.search(log.ts - 7 * day, log.ts, log_shape_id=shape.log_shape_id, limit=None))
speeds = reports.message.str.extract(r"\(\s*(\S+)\s+(\S+)\s+(\S+)\s+(\S+)\s+(\S+)\s+(\S+)\s*\)")
speeds.columns = [f"fan{i}" for i in range(1, 7)]
print(f"{len(reports)} reports, from {sorted(set(reports.node))}")
speeds.tail(5)

6110 reports, from ['Interconnect-0N00']


,fan1,fan2,fan3,fan4,fan5,fan6
6105,3552,3534,3375,****,3515,3479
6106,3552,3534,3375,****,3515,3479
6107,3552,3534,3375,****,3515,3479
6108,3552,3534,3375,****,3515,3479
6109,3552,3534,3375,****,3515,3479


Per fan: reports with no reading, and how many different values it showed

In [4]:
summary = pd.DataFrame({"no_reading": (speeds == "****").sum(), "distinct_values": speeds.nunique()})
display(summary)
assert summary.no_reading.drop("fan4").sum() == 0 and summary.no_reading.fan4 > 0.8 * len(reports)
assert (summary.distinct_values.drop("fan4") == 1).all()

,no_reading,distinct_values
fan1,0,1
fan2,0,1
fan3,0,1
fan4,5689,71
fan5,0,1
fan6,0,1


Maybe the 4th fan isn't working. I'd say we should alert the user with low severity.

Just to go the extra mile, we should check whether this affects the temperature of this device, has it overheated?

We dont know what temperature warnings look like so let's examine all logs on this component in the past day.

In [11]:
logs.node_timeline(log.node, log.ts - day, log.ts)

[TimelineEntry(component='switch_module', state='fan', template='Fan speeds ( <N> <N> <N> <N> <N> <N> )', log_shape_id=38, first_ts='2005-09-04 14:56:58', last_ts='2005-09-05 14:56:30', count=2833)]

# Verdict: not a clean report. Fan 4 of this switch, or its sensor, has no reading in most reports, and the other five fans never change.
This is the only switch that sends this log shape (fan speed reports). A hardware ticket, not urgent while the switch reports no high temperature.
The shape cannot tell this: six plain numbers would be the same shape. The value has to be read.